# Installations & Imports

The code will run by simply running the RUN ALL button, The GUI will spawn at the last cell all the way down, make sure to expand the GUI section

In [ ]:
!pip install requests beautifulsoup4
!pip install firebase
!pip install nltk

In [ ]:
import json
import re
from collections import Counter
import matplotlib.pyplot as plt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from nltk.stem import PorterStemmer
from firebase import firebase
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output, Javascript
from google.colab import drive
import io
import base64
from io import BytesIO
import seaborn as sns
import nltk
from nltk.chat.util import Chat, reflections

# DataBase


In [ ]:
def fetch_page(url):
   response = requests.get(url)
   if response.status_code == 200:
     soup = BeautifulSoup(response.text, 'html.parser')
     return soup
   else:
     return None

In [ ]:
def index_words(soup):
   index = {}
   words = re.findall(r'\w+', soup.get_text())
   for word in words:
     word = word.lower()
     if word in index:
       index[word] += 1
     else: index[word] = 1
   return index

In [ ]:
def remove_stop_words(index):
   stop_words = {'a', 'an', 'the', 'and', 'or','in', 'on', 'at', 'to', 'all', 'also', 'n', 'of'}
   for stop_word in stop_words:
     if stop_word in index:
       del index[stop_word]
   return index

In [ ]:
def apply_stemming(index):
   stemmer = PorterStemmer()
   stemmed_index = {}
   for word, count in index.items():
       stemmed_word = stemmer.stem(word)
       if stemmed_word in stemmed_index:
          stemmed_index[stemmed_word] += count
       else: stemmed_index[stemmed_word] = count
   return stemmed_index

In [ ]:
def search_engine(url, query):
   soup = fetch_page(url)
   if soup is None:
     return None
   index = index_words(soup)
   index = remove_stop_words(index)
   index = apply_stemming(index)
   results = search(query, index)
   return results

In [ ]:
def search(query, index):
   query_words = re.findall(r'\w+', query.lower())
   results = {}
   for word in query_words:
       if word in index:
         results[word] = index[word]
   return results

change the link if you need to

In [ ]:
FBconn = firebase.FirebaseApplication('https://hw2-onshape-default-rtdb.europe-west1.firebasedatabase.app', None)

**The three cells below leave commented**, should be uncommented only when you want to save a different json file to the one that is in Database

In [ ]:
# FBconn.delete('https://hw2-onshape-default-rtdb.europe-west1.firebasedatabase.app', None)

In [ ]:
# json_file_path = './jsonFile.json'

# # Read the JSON file
# with open(json_file_path, 'r') as json_file:
#     data = json.load(json_file)


# # Write to database
# result = FBconn.post('/jsonFile', data)

# # Get the data and print it
# res = FBconn.get('/jsonFile', None)

In [ ]:
# ### send top 10 words to firebase ###
# FBconn = firebase.FirebaseApplication('https://hw2-onshape-default-rtdb.europe-west1.firebasedatabase.app')

# url = 'https://cad.onshape.com/help/Content/Glossary/glossary.htm?tocpath=_____19'

# index = index_words(fetch_page(url))
# index = remove_stop_words(index)
# index = apply_stemming(index)
# top_10_words = dict(Counter(index).most_common(10))

# rank=1
# for word, count in top_10_words.items():
#    rank = rank*2/count
# rank = 1-rank
# print("Rank of the page is:",rank)
# new_dict = [{"term": key, "freq": value} for key, value in top_10_words.items()]

# result = FBconn.post('/Results', new_dict)
# print()
# print("Data saved in firebase in Results with name:",new_dict)

# Importing Data From Database

In this section we import data from the database and assign them to global variables for upcoming use.

In [ ]:
data = FBconn.get('/jsonFile', None) # This variable holds the raw data of the database
data_list = list(data.values())[0] # Represents the database data in a list format

# CSS

In [ ]:
### our css styling ###
css = """
<style>

    .dashboard-container {
        color: #3498db;
        background-color: #F5DEB3;
        margin: 20px;
        padding: 20px;
        border: 1px solid #052e16;
        border-radius: 5px;
        box-shadow: 2px 2px 10px rgba(0,0,0,0.1);
    }

    .nav-button {
        background-color: #3498db;
        border: 2px solid #2980b9;
        color: white;
        font-size: 14px;
        font-weight: bold;
        padding: 8px 16px;
        margin: 0 5px;
        cursor: pointer;
        border-radius: 15px;
        transition: all 0.3s ease;
        box-shadow: 0 2px 4px rgba(0,0,0,0.1);

        /* Center text both horizontally and vertically */
        display: flex;
        justify-content: center;
        align-items: center;
        text-align: center;

        /* Ensure consistent height */
        height: 36px;

        /* Prevent text wrapping */
        white-space: nowrap;
    }

    .nav-button:hover {
        background-color: #45a049;
    }

    #search-results {
        color: #3498db;
        margin-left: 10px;
    }
</style>

"""

# Dashboard Page

In [ ]:
# Function to save graph as an image
def save_graph_as_image(plt_figure, filename):
    plt_figure.savefig(filename, format='png')
    plt.close(plt_figure)

# Function to create a download button for the graph
def create_download_button(plt_figure, filename):
    btn = widgets.Button(description="Download")

    def on_button_click(b):
        buf = BytesIO()
        plt_figure.savefig(buf, format='png')
        buf.seek(0)
        b64 = base64.b64encode(buf.getvalue()).decode()
        js_code = f"""
        function downloadFile(filename, content) {{
            const blob = new Blob([Uint8Array.from(atob(content), c => c.charCodeAt(0))], {{type: 'image/png'}});
            const link = document.createElement('a');
            link.href = window.URL.createObjectURL(blob);
            link.download = filename;
            link.click();
            window.URL.revokeObjectURL(link.href);
        }}
        downloadFile("{filename}", "{b64}");
        """
        display(Javascript(js_code))

    btn.on_click(on_button_click)
    return btn

# Function to convert a Matplotlib figure to HTML
def plt_to_html(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    plt.close(fig)
    buf.seek(0)
    img_str = base64.b64encode(buf.getvalue()).decode('utf-8')
    return f'<img src="data:image/png;base64,{img_str}">'


# Function to create a collaboration graph
def create_collaboration_graph(df):
    df["Time"] = pd.to_datetime(df["Time"])
    df["Hour"] = df["Time"].dt.floor('H')
    collaborations = df.groupby(["Hour", "Document"]).User.nunique().reset_index()
    collaborations = collaborations[collaborations.User > 1]

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(collaborations["Hour"], collaborations["User"], width=0.1)
    ax.set_xlabel('Time')
    ax.set_ylabel('Number of Collaborations')
    ax.set_title('Collaborations Between Students Over Time')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()

    return fig, plt_to_html(fig)


# A line graph showing the total number of activities over time for the entire dataset.
def create_cumulative_activity_count(df):
    df["Time"] = pd.to_datetime(df["Time"])
    df = df.sort_values(by="Time")
    df["Cumulative Count"] = df.index + 1

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(df["Time"], df["Cumulative Count"])
    ax.set_xlabel('Time')
    ax.set_ylabel('Cumulative Count')
    ax.set_title('Cumulative Activity Count')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()

    return fig, plt_to_html(fig)


# Function to render the dashboard
def renderDashboard():
    df = pd.DataFrame(data_list)
    collaboration_fig, collaboration_graph = create_collaboration_graph(df)
    acitivity_fig, activity_graph = create_cumulative_activity_count(df)

    return [
        widgets.HTML("<h2>Dashboard</h2><p>Welcome to the dashboard. Here's an overview of the data:</p>"),
        widgets.HTML("<h3>Collaboration Graph</h3>"),
        widgets.HTML(collaboration_graph),
        widgets.HTML("<h3>Cumulative Activity Count</h3>"),
        widgets.HTML(activity_graph),
    ]


# Index Page

In [ ]:
page_container = widgets.VBox()  # Create a container for the widgets

def renderIndex():
    # Create a search box widget with a placeholder text
    search_box = widgets.Text(description="Search:", placeholder="Enter search term")

    # Create a widget to display search results, with some styling
    search_results = widgets.HTML(layout=widgets.Layout(margin='0 0 0 10px', color='blue'))

    # URL of the page to fetch and index
    url = 'https://cad.onshape.com/help/Content/Glossary/glossary.htm?tocpath=_____19'

    # Fetch and process the page content to build an index
    index = index_words(fetch_page(url))
    index = remove_stop_words(index)  # Remove common stop words
    index = apply_stemming(index)  # Apply stemming to the words

    # Get the top 10 most common words in the index
    top_10_words = dict(Counter(index).most_common(10))

    # Define a callback function that updates search results when the search box value changes
    def on_search_change(change):
        results = search(change.new, index)  # Search for the input term in the index
        if not results:
            result_text = f"{change.new}: 0"  # Display "0" if no results are found
        else:
            # Format the results as a list of word:count pairs
            result_text = "<br>".join([f"{word}: {count}" for word, count in results.items()])
        search_results.value = result_text  # Update the search results display

    # Attach the callback function to the search box's value change event
    search_box.observe(on_search_change, names='value')

    # Format the top 10 words as an HTML unordered list
    top_words_list = "<ul>" + "".join([f"<li>{word}: {count}</li>" for word, count in top_10_words.items()]) + "</ul>"

    # Layout the search box and results side by side
    search_layout = widgets.HBox([search_box, search_results])

    # Return the complete index page content as a list of widgets
    return [
        widgets.HTML("<h2>Index</h2><p>This is the index page. You can search or browse items here.</p>"),
        search_layout,
        widgets.HTML(top_words_list, layout=widgets.Layout(margin='10px 0 0 0', color='blue'))
    ]


# Filter Page

In [ ]:
def create_time_series_graph(user_df, user):
    # Count the number of actions per day
    date_counts = user_df["Time"].dt.date.value_counts().sort_index()

    # Create the plot
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(date_counts.index, date_counts.values, marker='o', linestyle='-')
    ax.set_xlabel('Date')
    ax.set_ylabel('Number of Actions')
    ax.set_title(f'Actions Over Time For: {user}')
    plt.xticks(rotation=45)
    plt.grid(True)

    # Return the figure and its HTML representation
    return fig, plt_to_html(fig)


def create_daily_distribution_graph(user_df, user):
    # Extract the hour from the timestamp
    user_df['hour'] = user_df["Time"].dt.hour

    # Create the histogram
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(user_df['hour'], bins=24, range=(0, 24), color='blue', alpha=0.7)
    ax.set_xlabel('Hour of the Day')
    ax.set_ylabel('Number of Actions')
    ax.set_title(f'Distribution of Actions Throughout the Day For: {user}')
    plt.grid(True)

    # Return the figure and its HTML representation
    return fig, plt_to_html(fig)


def create_timeframe_graph(df, selected_user, start_date, end_date):
    # Filter the dataframe for the selected time range
    filtered_df = df[(df['Time'] >= start_date) & (df['Time'] <= end_date)]

    # Calculate actions per day for the selected user
    user_df = filtered_df[filtered_df['User'] == selected_user]
    user_counts = user_df["Time"].dt.date.value_counts().sort_index()

    # Calculate sum of actions per day for all users
    all_users_counts = filtered_df["Time"].dt.date.value_counts().sort_index()

    # Ensure both series have the same date range
    date_range = pd.date_range(start=start_date, end=end_date)
    user_counts = user_counts.reindex(date_range, fill_value=0)
    all_users_counts = all_users_counts.reindex(date_range, fill_value=0)

    # Create the plot
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(user_counts.index, user_counts.values, marker='o', linestyle='-', color='blue', label=f'{selected_user} Actions')
    ax.plot(all_users_counts.index, all_users_counts.values, linestyle='--', color='red', label='All Users Actions')
    ax.set_xlabel('Date')
    ax.set_ylabel('Number of Actions')
    ax.set_title(f'Actions Comparison: {selected_user} vs All Users\n{start_date.date()} to {end_date.date()}')
    ax.tick_params(axis='x', rotation=45)
    ax.legend()
    ax.grid(True)

    plt.tight_layout()

    # Return the figure and its HTML representation
    return fig, plt_to_html(fig)

def create_activities_pie_chart(user_df, user):
    # Define keywords for different categories of activities
    keywords = {
        "creative": ["add", "modify", "edit", "insert", "create", "rename", "move", "copy", "paste",
                     "undo", "redo", "suppress", "unsuppress", "commit", "delete"],
        "viewing": ["open", "close", "view", "show", "hide", "change"],
        "administrative": ["update", "create"]
    }

    # Counter to keep track of activity counts
    activity_counts = Counter()

    # Categorize each action based on keywords
    for _, item in user_df.iterrows():
        description = item['Description'].lower()
        categorized = False
        for category, word_list in keywords.items():
            if any(word in description for word in word_list):
                activity_counts[category] += 1
                categorized = True
                break
        if not categorized:
            activity_counts['other'] += 1

    # Create pie chart
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.pie(activity_counts.values(), labels=activity_counts.keys(), autopct='%1.1f%%', startangle=90)
    ax.axis('equal')
    ax.set_title(f'Activities Distribution for {user}')

    return fig, plt_to_html(fig)

def on_user_change(change):
    selected_user = change.new
    if selected_user:
        df = pd.DataFrame(data_list)
        user_df = df[df['User'] == selected_user].copy()
        user_df["Time"] = pd.to_datetime(user_df["Time"])

        # Create and update graphs
        time_series_fig, time_series_graph = create_time_series_graph(user_df, selected_user)
        daily_distribution_fig, daily_distribution_graph = create_daily_distribution_graph(user_df, selected_user)
        activities_fig, activities_graph = create_activities_pie_chart(user_df, selected_user)

        time_series_widget.value = time_series_graph
        daily_distribution_widget.value = daily_distribution_graph
        activities_widget.value = activities_graph

        # Update download buttons
        time_series_download.children = [create_download_button(time_series_fig, f"{selected_user}_time_series.png")]
        daily_distribution_download.children = [create_download_button(daily_distribution_fig, f"{selected_user}_daily_distribution.png")]
        activities_download.children = [create_download_button(activities_fig, f"{selected_user}_activities.png")]

        # Update date range selector
        min_date = user_df["Time"].min().date()
        max_date = user_df["Time"].max().date()
        start_date_picker.min = min_date
        start_date_picker.max = max_date
        start_date_picker.value = min_date
        end_date_picker.min = min_date
        end_date_picker.max = max_date
        end_date_picker.value = max_date
    else:
        # Clear graphs and download buttons if no user is selected
        time_series_widget.value = ''
        daily_distribution_widget.value = ''
        timeframe_widget.value = ''
        activities_widget.value = ''
        time_series_download.children = []
        daily_distribution_download.children = []
        timeframe_download.children = []
        activities_download.children = []


def on_date_change(change):
    selected_user = user_dropdown.value
    start_date = start_date_picker.value
    end_date = end_date_picker.value

    if selected_user and start_date and end_date:
        df = pd.DataFrame(data_list)
        df["Time"] = pd.to_datetime(df["Time"])

        # Create and update timeframe graph
        timeframe_fig, timeframe_graph = create_timeframe_graph(df, selected_user, pd.Timestamp(start_date), pd.Timestamp(end_date))
        timeframe_widget.value = timeframe_graph
        timeframe_download.children = [create_download_button(timeframe_fig, f"{selected_user}_timeframe.png")]
    else:
        # Clear timeframe graph and download button if data is incomplete
        timeframe_widget.value = ''
        timeframe_download.children = []


def renderFilter():
    users = list(set(item['User'] for item in data_list))
    users.sort()

    # Create global widget variables
    global user_dropdown, start_date_picker, end_date_picker, time_series_widget, daily_distribution_widget, timeframe_widget, activities_widget
    global time_series_download, daily_distribution_download, timeframe_download, activities_download

    # Create user dropdown
    user_dropdown = widgets.Dropdown(options=[('SELECT', None)] + [(user, user) for user in users], description='Select User:')
    user_dropdown.observe(on_user_change, names='value')

    # Create date pickers
    start_date_picker = widgets.DatePicker(description='Start Date')
    end_date_picker = widgets.DatePicker(description='End Date')

    start_date_picker.observe(on_date_change, names='value')
    end_date_picker.observe(on_date_change, names='value')

    # Create HTML widgets for graphs
    time_series_widget = widgets.HTML()
    daily_distribution_widget = widgets.HTML()
    timeframe_widget = widgets.HTML()
    activities_widget = widgets.HTML()

    # Create containers for download buttons
    time_series_download = widgets.HBox()
    daily_distribution_download = widgets.HBox()
    timeframe_download = widgets.HBox()
    activities_download = widgets.HBox()

    # Return the layout of the filter section
    return [
        widgets.HTML("<h2>Filter</h2><p>Select a user from the dropdown below:</p>"),
        user_dropdown,
        widgets.HTML("<h3>Overall Actions Over Time</h3>"),
        time_series_widget,
        time_series_download,
        widgets.HTML("<h3>Overall Distribution of Actions Throughout the Day</h3>"),
        daily_distribution_widget,
        daily_distribution_download,
        widgets.HTML("<h3>Activities Distribution</h3>"),
        activities_widget,
        activities_download,
        widgets.HTML("<h3>Actions within Selected Timeframe</h3>"),
        widgets.HBox([start_date_picker, end_date_picker]),
        timeframe_widget,
        timeframe_download
    ]

# GUI



Please wait a second between each page since it is pretty slow, The three buttons Dashboard-Filter-Index are our 3 pages


In [ ]:
# Create the container to hold the pages
page_container = widgets.VBox()

# Function to switch pages
def switch_page(page):
    if page == 'Dashboard':
      page_container.children = renderDashboard()
    elif page == 'Filter':
      page_container.children = renderFilter()
    elif page == 'Index':
      page_container.children = renderIndex()

# Create navigation menu
nav_menu = widgets.HBox([
    widgets.Button(description="Dashboard", layout=widgets.Layout(width='auto')),
    widgets.Button(description="Filter", layout=widgets.Layout(width='auto')),
    widgets.Button(description="Index", layout=widgets.Layout(width='auto'))
])

# Add a class to each navigation button
for button in nav_menu.children:
    button.add_class('nav-button')

# Function to handle button clicks
def on_button_click(b):
    switch_page(b.description)

# Assign click handlers to buttons
for button in nav_menu.children:
    button.on_click(on_button_click)


# Display the CSS
display(HTML(css))

# Main title
main_title = widgets.HTML("<h1>OnShape Data Analysis</h1>")


# Create a container for the entire dashboard
dashboard_container = widgets.VBox([main_title, nav_menu, page_container])

# Apply the 'dashboard-container' class to the main container
dashboard_container.add_class('dashboard-container')

# Display the layout
display(dashboard_container)

# Initialize with Dashboard page
switch_page('Dashboard')

# ChatBot

In [ ]:
# Retrieve data from Firebase
dbData = FBconn.get('/jsonFile', None)
# Extract the first value from the dictionary
data1 = list(dbData.values())[0]
# Create a list of unique users from the data
users = list(set(item['User'] for item in data1))



def whoWorkedTheMost():
    # Dictionary to store the number of actions per user
    user_actions = {}

    # Count actions for each user
    for item in data1:
        user = item['User']
        user_actions[user] = user_actions.get(user, 0) + 1

    # If no actions were recorded, return an empty list
    if not user_actions:
        return []

    # Find the maximum number of actions
    max_actions = max(user_actions.values())
    # Find all users who performed the maximum number of actions
    top_users = [user for user, actions in user_actions.items() if actions == max_actions]

    # Return a formatted string with the result
    return f'The user(s) with the most actions is/are: {top_users}, they made {max_actions} actions'

def projectName():
    # Return the name of the document from the first data entry
    return f'The project name is: {data1[0]["Document"]}'

def totalAction():
    # Return the total number of actions (entries in the data)
    return f'The total number of actions is: {len(data1)}'


In [ ]:
# Download necessary NLTK data
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)

# Cloud explanation text
CLOUD_EXPLANATION = """
Private Cloud: A private cloud is a cloud computing environment dedicated solely to one organization.
It's usually managed by the organization itself or a third-party provider, offering greater control,
security, and customization options.

Public Cloud: A public cloud is hosted and managed by a third-party service provider and made available
to multiple organizations or individuals over the internet. It offers scalability, flexibility, and
cost-effectiveness.

Hybrid Cloud: A hybrid cloud combines elements of both private and public clouds. It allows organizations
to leverage benefits of both environments by integrating on-premises infrastructure with public cloud
resources.
"""

# Glossary terms and their explanations
GLOSSARY_TERMS = {
    "CAD": "Computer-Aided Design (CAD) is the use of computer systems to assist in the creation, modification, analysis, or optimization of a design.",
    "Sketch": "A sketch in OnShape is a 2D drawing that forms the basis for creating 3D models.",
    "Extrude": "Extrude is a feature that adds depth to a 2D sketch, creating a 3D object.",
    "Assembly": "An assembly in OnShape is a collection of parts and subassemblies that are combined to create a final product.",
    "Part": "A part in OnShape is a single 3D object created from one or more sketches.",
    "Feature": "A feature in OnShape is a tool or function used to create or modify parts, such as extrude, revolve, or fillet."
}

# Function to handle special queries
def handle_special_query(query):
    if query == 'who worked the most?':
        return whoWorkedTheMost()
    elif query in ['what document are you working on?', 'what is the name of the project?']:
        return projectName()
    elif query == 'what is the total number of actions?':
        return totalAction()
    return None

# Chat patterns and responses
PATTERNS = [
    (r'hi|hello|hey', ['Hello!', 'Hi there!', 'Hey!']),
    (r'how are you?', ['I\'m good, thank you!', 'I\'m doing well, thanks for asking.']),
    (r'what is your name?', ['You can call me onShapeBot.', 'I go by the name onShapeBot.']),
    (r'my name is (.*)', ['Nice to meet you, %1.']),
    (r'thank you (.*)', ['You\'re welcome!', 'Happy to help!']),
    (r'what is the purpose of the system?', ['I can show you interesting statistics about your employees.']),
    (r'what is cloud|cloud', [CLOUD_EXPLANATION]),
    (r'what is onshape|onshape|on shape', ['Onshape is a cloud-based CAD platform for product design and development. Learn more here: https://www.onshape.com/']),
    (r'who worked the most?|what document are you working on?|what is the name of the project?|what is the total number of actions?', ['Let me check that for you.']),
    (r'bye|goodbye|quit', ['Goodbye!', 'See you later!'])
]

# Add patterns for glossary terms
for term, explanation in GLOSSARY_TERMS.items():
    PATTERNS.extend([
        (fr'what is {term.lower()}', [explanation]),
        (fr'can you explain {term.lower()}', [explanation])
    ])

def create_chatbot():
    """Create and return a chatbot instance."""
    return Chat(PATTERNS, reflections)

def run_chat():
    """Run the chatbot conversation."""
    chatbot = create_chatbot()
    print("Hello! I'm onShapeBot. How can I help you today?")

    while True:
        user_input = input("You: ")
        if user_input.lower() in ['exit', 'bye', 'quit', 'goodbye']:
            print("onShapeBot: Goodbye!")
            break

        response = chatbot.respond(user_input)

        # Check for special queries
        special_response = handle_special_query(user_input.lower())
        if special_response:
            response = special_response

        print("onShapeBot:", response)

if __name__ == "__main__":
    run_chat()

Hello! I'm onShapeBot. How can I help you today?
You: quit
onShapeBot: Goodbye!
